In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import coo_matrix, save_npz

In [2]:
# Carga del dataset de Music Info: 

path_music_info = '../data/raw/Music Info.csv'

use_cols = ['track_id', 'name', 'artist', 'tags', 'genre', 'year', 'duration_ms', ]
df_music_info = pd.read_csv(path_music_info)
df_music_info.info()

<class 'pandas.DataFrame'>
RangeIndex: 50683 entries, 0 to 50682
Data columns (total 21 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   track_id             50683 non-null  str    
 1   name                 50683 non-null  str    
 2   artist               50683 non-null  str    
 3   spotify_preview_url  50683 non-null  str    
 4   spotify_id           50683 non-null  str    
 5   tags                 49556 non-null  str    
 6   genre                22348 non-null  str    
 7   year                 50683 non-null  int64  
 8   duration_ms          50683 non-null  int64  
 9   danceability         50683 non-null  float64
 10  energy               50683 non-null  float64
 11  key                  50683 non-null  int64  
 12  loudness             50683 non-null  float64
 13  mode                 50683 non-null  int64  
 14  speechiness          50683 non-null  float64
 15  acousticness         50683 non-null  float64
 1

In [3]:
# Carga del dataset de Listening History: 

path_history = '../data/raw/User Listening History.csv'

dtypes = {'track_id': 'category', 'user_id': 'category', 'playcount': 'uint16'}
df_history = pd.read_csv(path_history, dtype=dtypes)
df_history.info()

<class 'pandas.DataFrame'>
RangeIndex: 9711301 entries, 0 to 9711300
Data columns (total 3 columns):
 #   Column     Dtype   
---  ------     -----   
 0   track_id   category
 1   user_id    category
 2   playcount  uint16  
dtypes: category(2), uint16(1)
memory usage: 114.9 MB


In [ ]:
# Creamos la matriz de usuarios-ítems. Puesto que ello requiere hace un pivot_table que
# generaría una matriz de tamaño n_usuarios x n_ítems, lo cual satura fácilmente la RAM, 
# necesitamos crear una estructura que gestione eficientemente matrices dispersas. En nuestro 
# caso hemos decidido usar COO de Scipy por sencillez, que luego transformaremos a CSR por su 
# facilidad con el cálculo matricial: 

# En primer lugar, usamos factorize de pandas para usar códigos como enteros que comiencen en 0
# que se correspondan con sus respectivos ids: 
u_codes, u_ids = pd.factorize(df_history.user_id, sort=True)
i_codes, i_ids = pd.factorize(df_history.track_id, sort=True)

playcounts = df_history.playcount
ui_matrix = coo_matrix((playcounts, (u_codes, i_codes)), 
                       shape=(len(u_ids), len(i_ids))).tocsr().astype('uint16')

# Guardamos la matriz en formato npz para poder reutilizarla en otros documentos: 
save_npz('../data/processed/ui_matrix_csr.npz', ui_matrix)